# Workbook export, step by step

This notebook is the slow, inspectable counterpart to `export_excel_to_csv.py`. It follows the same order, transformations, output schema, and diagnostics, with checkpoints and plots for manual review.

In [ ]:
from collections import Counter, defaultdict
from datetime import datetime, timedelta
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
EXCEL_PATH = REPO_ROOT / "data_exel/cascadia_dashboard_master_updated_06-15-26.xlsx"
NODES_SHEET, EDGES_SHEET = "master", "Relationships"
OUT_DIR = REPO_ROOT / "scripts/out"
NODES_OUT = OUT_DIR / "organizations_clean.csv"
EDGES_OUT = OUT_DIR / "edges_clean.csv"
REPORT_OUT = OUT_DIR / "export_excel_report.json"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Workbook:", EXCEL_PATH, "| exists:", EXCEL_PATH.exists())


## 1. Constants and helpers copied from the script

In [ ]:
MISSING_TOKENS = {"", "nan", "none", "n/a", "na", "null", "???"}
CATEGORY_EXPORTS = {
    "node_type": ("nodeTypes_json", "nodeTypePrimary"),
    "org_type": ("orgTypes_json", "orgTypePrimary"),
    "governance_level": ("governanceLevels_json", "governanceLevelPrimary"),
    "geographic_scale": ("geoTags_json", "geoPrimary"),
    "roles": ("roleTags_json", "rolePrimary"),
}
NODE_COLUMNS = [
    "Organization Name", "Org ID", "orgTypes_json", "orgTypePrimary",
    "geoPrimary", "Notes", "Primary", "2ndry", "geoTags_json",
    "nodeTypes_json", "nodeTypePrimary", "governanceLevels_json",
    "governanceLevelPrimary", "roleTags_json", "rolePrimary", "url",
    "review_flag", "review_note", "lastUpdated",
]
EDGE_COLUMNS = ["From agency", "To agency", "Relationship type", "Description", "Status"]

def normalize_text(value):
    value = re.sub(r"\s+", " ", str(value if pd.notna(value) else "")).strip()
    return "" if value.lower() in MISSING_TOKENS else value

def clean_id(value):
    return re.sub(r"\s+", "", normalize_text(value))

def pretty_label(value):
    value = normalize_text(value)
    if not value: return "Other"
    if "_" in value: return value.replace("_", " ").title()
    return value.title() if value.islower() else value

def split_categories(value):
    value = normalize_text(value)
    if not value: return ["Other"]
    categories = []
    # Multi-value fields are comma-separated; slashes occur inside valid labels.
    for part in re.split(r"\s*,\s*", value):
        label = pretty_label(part)
        if label not in categories: categories.append(label)
    return categories or ["Other"]

def excel_date(value):
    # pandas may decode a formatted Excel serial into a Timestamp; the script sees the raw serial.
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.date().isoformat()
    value = normalize_text(value)
    if not value: return ""
    try: serial = float(value)
    except ValueError: return value
    if not 1 <= serial <= 2_958_465: return value
    return (datetime(1899, 12, 30) + timedelta(days=serial)).date().isoformat()

def combine_notes(row):
    parts = []
    for column in ("Summary", "Description of relevant resources"):
        value = normalize_text(row.get(column))
        if value and value not in parts: parts.append(value)
    return "\n\n".join(parts)


## 2. Read the workbook and validate both sheet schemas

In [ ]:
node_source = pd.read_excel(EXCEL_PATH, sheet_name=NODES_SHEET, dtype=object)
edge_source = pd.read_excel(EXCEL_PATH, sheet_name=EDGES_SHEET, dtype=object)
print(f"{NODES_SHEET}: {node_source.shape}")
display(node_source.head())
print(f"{EDGES_SHEET}: {edge_source.shape}")
display(edge_source.head())

required_nodes = ["node_id", "name", *CATEGORY_EXPORTS]
required_edges = ["From agency", "To agency", "Relationship type"]
missing_nodes = [c for c in required_nodes if c not in node_source.columns]
missing_edges = [c for c in required_edges if c not in edge_source.columns]
assert not missing_nodes, f"Sheet {NODES_SHEET!r} is missing: {', '.join(missing_nodes)}"
assert not missing_edges, f"Sheet {EDGES_SHEET!r} is missing: {', '.join(missing_edges)}"


## 3. Clean and manually check node IDs

In [ ]:
node_work = node_source.copy()
node_work["_source_row"] = range(2, len(node_work) + 2)
node_work["_clean_id"] = node_work["node_id"].apply(clean_id)
skipped_nodes = node_work.loc[node_work["_clean_id"].eq(""), ["_source_row", "name"]].copy()
skipped_nodes.columns = ["row", "name"]
skipped_nodes["name"] = skipped_nodes["name"].apply(normalize_text)
print("Rows skipped for missing node_id:", len(skipped_nodes))
display(skipped_nodes)

node_work = node_work.loc[node_work["_clean_id"].ne("")].copy()
duplicate_ids = node_work.loc[node_work["_clean_id"].duplicated(False)].sort_values("_clean_id")
print("Rows with duplicate cleaned node_id:", len(duplicate_ids))
display(duplicate_ids[["_source_row", "node_id", "_clean_id", "name"]])
assert duplicate_ids.empty, "The exporter stops on duplicate node IDs; resolve these first."


## 4. Inspect categorical values before normalization

In [ ]:
for source in CATEGORY_EXPORTS:
    counts = node_work[source].apply(normalize_text).replace("", "(missing)").value_counts()
    print(f"{source}: {len(counts)} distinct values")
    display(counts.rename("rows").to_frame().head(30))


## 5. Build node output, one source row at a time

In [ ]:
category_counts = defaultdict(Counter)
category_normalizations = defaultdict(Counter)
node_rows = []
for _, row in node_work.iterrows():
    item = {
        "Organization Name": normalize_text(row.get("name")),
        "Org ID": row["_clean_id"],
        "Notes": combine_notes(row),
        "Primary": normalize_text(row.get("key_contact")),
        "2ndry": normalize_text(row.get("contact_email")) or normalize_text(row.get("contact_url")),
        "url": normalize_text(row.get("url")),
        "review_flag": normalize_text(row.get("review_flag")),
        "review_note": normalize_text(row.get("review_note")),
        "lastUpdated": excel_date(row.get("last_update")),
    }
    for source, (list_col, primary_col) in CATEGORY_EXPORTS.items():
        categories = split_categories(row.get(source))
        item[list_col], item[primary_col] = json.dumps(categories), categories[0]
        raw, normalized = normalize_text(row.get(source)), ", ".join(categories)
        category_counts[source].update(categories)
        if raw != normalized:
            category_normalizations[source][(raw, normalized)] += 1
    node_rows.append(item)

clean_df = pd.DataFrame(node_rows, columns=NODE_COLUMNS)
print(f"Source: {len(node_source)} | exported: {len(clean_df)}")
display(clean_df.head(10))


In [ ]:
normalization_df = pd.DataFrame([
    {"field": source, "from": before, "to": after, "rows": count}
    for source, changes in category_normalizations.items()
    for (before, after), count in sorted(changes.items())
], columns=["field", "from", "to", "rows"])
display(normalization_df)

fig, axes = plt.subplots(len(CATEGORY_EXPORTS), 1, figsize=(12, 15))
for ax, source in zip(axes, CATEGORY_EXPORTS):
    pd.Series(dict(category_counts[source])).sort_values().plot.barh(
        ax=ax, title=f"Normalized {source}"
    )
    ax.set_xlabel("organizations")
plt.tight_layout()


## 6. Build edges with the script's exact five-column duplicate signature

In [ ]:
edge_rows, duplicate_edges, blank_edge_rows = [], [], []
seen_edges = set()
for source_row, (_, row) in enumerate(edge_source.iterrows(), start=2):
    item = {column: normalize_text(row.get(column)) for column in EDGE_COLUMNS}
    item["From agency"] = clean_id(item["From agency"])
    item["To agency"] = clean_id(item["To agency"])
    signature = tuple(item[column] for column in EDGE_COLUMNS)
    if not any(signature):
        blank_edge_rows.append(source_row)
    elif signature in seen_edges:
        duplicate_edges.append({"row": source_row, **item})
    else:
        seen_edges.add(signature)
        edge_rows.append(item)

df_edges_clean = pd.DataFrame(edge_rows, columns=EDGE_COLUMNS)
print(f"Source: {len(edge_source)} | exported: {len(df_edges_clean)} | duplicates: {len(duplicate_edges)}")
print("Blank rows:", blank_edge_rows)
display(pd.DataFrame(duplicate_edges, columns=["row", *EDGE_COLUMNS]))
display(df_edges_clean.head(10))


## 7. Relationship diagnostics and coverage visualizations

In [ ]:
node_ids = set(clean_df["Org ID"])
from_ids = {x for x in df_edges_clean["From agency"] if x}
to_ids = {x for x in df_edges_clean["To agency"] if x}
endpoint_ids = from_ids | to_ids
relationships = {
    "edgeSourcesMissingFromNodes": sorted(from_ids - node_ids),
    "edgeTargetsMissingFromNodes": sorted(to_ids - node_ids),
    "organizationsWithRelationships": sorted(node_ids & endpoint_ids),
    "organizationsWithoutRelationships": sorted(node_ids - endpoint_ids),
}
print("Missing sources:", relationships["edgeSourcesMissingFromNodes"])
print("Missing targets:", relationships["edgeTargetsMissingFromNodes"])
print("Organizations without relationships:", relationships["organizationsWithoutRelationships"])

coverage = pd.DataFrame({
    "status": ["has at least one relationship", "has no relationships"],
    "count": [len(relationships["organizationsWithRelationships"]),
              len(relationships["organizationsWithoutRelationships"])],
})
coverage["share"] = coverage["count"] / coverage["count"].sum() if coverage["count"].sum() else 0
display(coverage)
coverage.set_index("status")["count"].plot.barh(figsize=(8, 3), title="Relationship coverage")
plt.xlabel("organizations")
plt.tight_layout()

display(clean_df.assign(has_relationship=clean_df["Org ID"].isin(endpoint_ids))
        .sort_values(["has_relationship", "Org ID"], ascending=[False, True])
        [["Org ID", "Organization Name", "orgTypePrimary", "geoPrimary", "has_relationship"]]
        .reset_index(drop=True))


## 8. Assemble the same JSON diagnostics report

In [ ]:
report = {
    "sourceWorkbook": str(EXCEL_PATH.relative_to(REPO_ROOT)).replace("\\", "/"),
    "nodes": {
        "sourceRows": len(node_source),
        "exportedRows": len(clean_df),
        "skippedMissingNodeIds": skipped_nodes.to_dict(orient="records"),
        "categoryCounts": {s: dict(sorted(c.items())) for s, c in category_counts.items()},
        "categoryNormalizations": {
            s: [{"from": a, "to": b, "rows": n} for (a, b), n in sorted(changes.items())]
            for s, changes in category_normalizations.items()
        },
    },
    "edges": {
        "sourceRows": len(edge_source),
        "exportedRows": len(df_edges_clean),
        "duplicateRowsRemoved": duplicate_edges,
        "blankRowsSkipped": blank_edge_rows,
    },
    "relationships": relationships,
}
display(report)


## 9. Export

Writes are isolated in the final cell so all earlier cells can be used for manual checking first.

In [ ]:
clean_df.to_csv(NODES_OUT, index=False)
df_edges_clean.to_csv(EDGES_OUT, index=False)
REPORT_OUT.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Wrote {NODES_OUT} ({len(clean_df)} nodes)")
print(f"Wrote {EDGES_OUT} ({len(df_edges_clean)} edges)")
print(f"Wrote {REPORT_OUT}")
